# Problem: Implement Logistic Regression

### Problem Statement
Your task is to implement a **Logistic Regression** model using PyTorch. The model should predict a binary class label based on a given set of input features.

### Requirements
1. **Model Definition**:
   - Implement a class `LogisticRegressionModel` with:
     - A single linear layer mapping input features to one output.
     - A sigmoid activation to produce a probability.
2. **Forward Method**:
   - Implement the `forward` method to compute predictions given input data.

### Part 1: Using `nn.Module`
### Part 2: Using Raw Tensor Operations

In [5]:
import torch
import torch.nn as nn
import torch.optim as optim

In [6]:
# Generate synthetic binary classification data
torch.manual_seed(42)
X = torch.randn(200, 2)  # 200 data points, 2 features
y = ((X[:, 0] + X[:, 1]) > 0).float().unsqueeze(1)  # label 1 if sum > 0

## Part 1: Logistic Regression with `nn.Module`

In [7]:
# Define the Logistic Regression Model
class LogisticRegressionModel(nn.Module):
    def __init__(self):
        super(LogisticRegressionModel, self).__init__()
        self.linear = nn.Linear(2, 1)

    def forward(self, x):
        return torch.sigmoid(self.linear(x))

# Initialize the model, loss function, and optimizer
model = LogisticRegressionModel()
criterion = nn.BCELoss()
optimizer = optim.SGD(model.parameters(), lr=0.1)

# Training loop
epochs = 1000
for epoch in range(epochs):
    # Forward pass
    predictions = model(X)
    loss = criterion(predictions, y)

    # Backward pass and optimization
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if (epoch + 1) % 100 == 0:
        print(f"Epoch [{epoch + 1}/{epochs}], Loss: {loss.item():.4f}")

Epoch [100/1000], Loss: 0.2783
Epoch [200/1000], Loss: 0.2171
Epoch [300/1000], Loss: 0.1877
Epoch [400/1000], Loss: 0.1694
Epoch [500/1000], Loss: 0.1564
Epoch [600/1000], Loss: 0.1467
Epoch [700/1000], Loss: 0.1389
Epoch [800/1000], Loss: 0.1325
Epoch [900/1000], Loss: 0.1272
Epoch [1000/1000], Loss: 0.1226


In [8]:
# Display the learned parameters
[w, b] = model.linear.parameters()
print(f"Learned weights: {w.data}, Learned bias: {b.item():.4f}")

# Compute accuracy
with torch.no_grad():
    preds = (model(X) >= 0.5).float()
    accuracy = (preds == y).float().mean()
    print(f"Training accuracy: {accuracy.item():.4f}")

# Test on new data
X_test = torch.tensor([[1.0, 1.0], [-1.0, -1.0]])
with torch.no_grad():
    probs = model(X_test)
    print(f"Predictions for {X_test.tolist()}: {probs.tolist()}")

Learned weights: tensor([[3.7889, 3.7170]]), Learned bias: -0.1590
Training accuracy: 0.9950
Predictions for [[1.0, 1.0], [-1.0, -1.0]]: [[0.9993558526039124], [0.00046875860425643623]]


## Part 2: Logistic Regression with Raw Tensor Operations

Implement the same model **without** `nn.Module`, `nn.Linear`, or `optim`. Use only tensors, sigmoid, and autograd.

In [9]:
# Manual parameter initialization (reuse X, y from above)
w = torch.randn(2, 1, requires_grad=True)
b = torch.zeros(1, requires_grad=True)

# Training loop
lr = 0.1
epochs = 1000
for epoch in range(epochs):
    # Forward pass with sigmoid
    y_pred = torch.sigmoid(X @ w + b)

    # Binary cross-entropy loss
    loss = -(y * torch.log(y_pred + 1e-7) + (1 - y) * torch.log(1 - y_pred + 1e-7)).mean()

    # Backward pass
    loss.backward()

    # Update parameters manually
    with torch.no_grad():
        w -= lr * w.grad
        b -= lr * b.grad

    # Zero gradients
    w.grad.zero_()
    b.grad.zero_()

    if (epoch + 1) % 100 == 0:
        print(f"Epoch [{epoch + 1}/{epochs}], Loss: {loss.item():.4f}")

Epoch [100/1000], Loss: 0.2561
Epoch [200/1000], Loss: 0.2076
Epoch [300/1000], Loss: 0.1821
Epoch [400/1000], Loss: 0.1655
Epoch [500/1000], Loss: 0.1536
Epoch [600/1000], Loss: 0.1444
Epoch [700/1000], Loss: 0.1371
Epoch [800/1000], Loss: 0.1310
Epoch [900/1000], Loss: 0.1259
Epoch [1000/1000], Loss: 0.1215


In [10]:
# Display the learned parameters
print(f"Learned weights: {w.data.squeeze().tolist()}, Learned bias: {b.item():.4f}")

# Compute accuracy
with torch.no_grad():
    preds = (torch.sigmoid(X @ w + b) >= 0.5).float()
    accuracy = (preds == y).float().mean()
    print(f"Training accuracy: {accuracy.item():.4f}")

# Test on new data
X_test = torch.tensor([[1.0, 1.0], [-1.0, -1.0]])
with torch.no_grad():
    probs = torch.sigmoid(X_test @ w + b)
    print(f"Predictions for {X_test.tolist()}: {probs.tolist()}")

Learned weights: [3.8268911838531494, 3.7561049461364746], Learned bias: -0.1616
Training accuracy: 0.9950
Predictions for [[1.0, 1.0], [-1.0, -1.0]]: [[0.9994020462036133], [0.0004328814393375069]]
